In [1]:
import os
import random
import glob

In [2]:
def _extract_key(path: str, key_type: str = "subject") -> str:
    assert key_type in ["subject", "id"], f"Invalid key type '{key_type}'. Choose from: ['subject', 'id']"
    
    norm = path.replace("\\", "/")

    if "/FVC/" in norm:
        """
        'data/FVC/fvc2000/db1/100_1.tif' → 'fvc2000_db1_100'
        """
        parts  = norm.split("/")
        year   = parts[-3]         
        db     = parts[-2]
        id = os.path.basename(norm).split("_")[0]
        return f"{year}_{db}_{id}"
    
    if "/SD302/" in norm:
        parts = os.path.basename(norm).split("_")
        subject = parts[0]
        id = os.path.splitext(parts[-1])[0]
        if key_type == "subject":
            """
            'data/SD302/a/A/00002500_A_roll_01.png'     → 'sd302_00002500'
            'data/SD302/b/U/00002546_U_500_roll_07.png' → 'sd302_00002546'
            """
            return f"sd302_{subject}"
        else:   # "id"
            """
            'data/SD302/a/A/00002500_A_roll_01.png'     → 'sd302_00002500_01'
            """
            return f"sd302_{subject}_{id}"
    
    if "/LivDet/" in norm:
        parts      = norm.split("/")
        livdet_idx = parts.index("LivDet")
        year       = parts[livdet_idx + 1]
        sensor     = parts[livdet_idx + 2]

        stem       = os.path.splitext(parts[-1])[0]
        tokens     = stem.split("_")

        if len(tokens) == 3:    # subject_finger_impression
            subject, finger, _ = tokens
            if key_type == "subject":
                """
                'data/LivDet/livdet2015/CrossMatch/Train/Live/0310542_L1_1.bmp' → 'livdet2015_CrossMatch_0310542'
                """
                return f"{year}_{sensor}_{subject}"
            else:   # "id"
                """
                'data/LivDet/livdet2015/CrossMatch/Train/Live/0310542_L1_1.bmp' → 'livdet2015_CrossMatch_0310542_L1'
                """
                return f"{year}_{sensor}_{subject}_{finger}"

        else:  # id_impression
            id_, impression = tokens
            """
            'data/LivDet/livdet2011/Biometrika/Train/Live/1_1.png' → 'livdet2011_Biometrika_1'
            """
            return f"{year}_{sensor}_{id_}"
    
    raise ValueError(f"Cannot determine dataset for path '{path}'.")

In [3]:
data_root = "data/SD302/"

modalities = {
    "sd302a_A": os.path.join(data_root, "sd302a", "A"),
    "sd302a_B": os.path.join(data_root, "sd302a", "B"),
    "sd302a_C": os.path.join(data_root, "sd302a", "C"),
    "sd302a_E": os.path.join(data_root, "sd302a", "E"),
    "sd302a_F": os.path.join(data_root, "sd302a", "F"),
    "sd302a_G": os.path.join(data_root, "sd302a", "G"),
    "sd302b_R": os.path.join(data_root, "sd302b", "R"),
    "sd302b_S": os.path.join(data_root, "sd302b", "S"),
    "sd302b_U": os.path.join(data_root, "sd302b", "U"),
    "sd302b_V": os.path.join(data_root, "sd302b", "V"),
    "sd302c_J": os.path.join(data_root, "sd302c", "J"),
    "sd302c_N": os.path.join(data_root, "sd302c", "N"),
    "sd302c_Q": os.path.join(data_root, "sd302c", "Q"),
    "sd302d_K": os.path.join(data_root, "sd302d", "K"),
    "sd302d_L": os.path.join(data_root, "sd302d", "L"),
    "sd302d_M": os.path.join(data_root, "sd302d", "M"),
    "sd302d_P": os.path.join(data_root, "sd302d", "P")
}

subject_to_paths: dict[str, list[str]] = {}
id_to_paths: dict[str, list[str]] = {}

for modal_key, modal_dir in modalities.items():
    for path in sorted(glob.glob(os.path.join(modal_dir, "*.png"))):
        id_to_paths.setdefault(_extract_key(path, "id"), []).append(path)
        subject_to_paths.setdefault(_extract_key(path, "subject"), []).append(path)

subject_counts = {sid: len(paths) for sid, paths in subject_to_paths.items()}
id_counts = {id_: len(paths) for id_, paths in id_to_paths.items()}

max_subject = max(subject_counts, key=subject_counts.get)
min_subject = min(subject_counts, key=subject_counts.get)

max_id = max(id_counts, key=id_counts.get)
min_id = min(id_counts, key=id_counts.get)

max_subject_count = subject_counts[max_subject]
min_subject_count = subject_counts[min_subject]

max_id_count = id_counts[max_id]
min_id_count = id_counts[min_id]

print(f"Max paths: {max_subject_count} (Subject: {max_subject})")
print(f"Min paths: {min_subject_count} (Subject: {min_subject})")
print()
print(f"Max paths: {max_id_count} (Id: {max_id})")
print(f"Min paths: {min_id_count} (Id: {min_id})")

Max paths: 144 (Subject: sd302_00002445)
Min paths: 98 (Subject: sd302_00002361)

Max paths: 15 (Id: sd302_00002437_03)
Min paths: 8 (Id: sd302_00002564_06)


In [6]:
for subject, path in subject_to_paths.items():
    break

from PIL import Image
img = Image.open(path[3])
img.size

(512, 512)

In [8]:
min_samples = 3
max_samples = 20

def _collect_live_paths(sensor_dir: str, split: str) -> list[str]:
    base = os.path.join(sensor_dir, split, "Live")
    return sorted(glob.glob(os.path.join(base, "*.png")) + glob.glob(os.path.join(base, "*.bmp")))

def _filter_by_id(paths: list[str], rng: random.Random) -> list[str]:
    id_to_paths: dict[str, list[str]] = {}
    for p in paths:
        id_to_paths.setdefault(_extract_key(p, "id"), []).append(p)

    kept: list[str] = []
    for id_paths in id_to_paths.values():
        if len(id_paths) < min_samples:
            continue
        if len(id_paths) > max_samples:
            id_paths = rng.sample(id_paths, max_samples)
        kept.extend(id_paths)

    return sorted(kept)

rng = random.Random(42)

data_root:   str   = "data/LivDet/"
sensors = {
    "livdet2011_Biometrika"     : os.path.join(data_root, "livdet2011", "Biometrika"),
    "livdet2011_Digital"        : os.path.join(data_root, "livdet2011", "Digital"),
    "livdet2011_Italdata"       : os.path.join(data_root, "livdet2011", "Italdata"),
    "livdet2011_Sagem"          : os.path.join(data_root, "livdet2011", "Sagem"),
    "livdet2013_Biometrika"     : os.path.join(data_root, "livdet2013", "Biometrika"),
    "livdet2013_CrossMatch"     : os.path.join(data_root, "livdet2013", "CrossMatch"),
    "livdet2013_Italdata"       : os.path.join(data_root, "livdet2013", "Italdata"),
    "livdet2015_CrossMatch"     : os.path.join(data_root, "livdet2015", "CrossMatch"),
    "livdet2015_DigitalPersona" : os.path.join(data_root, "livdet2015", "DigitalPersona"),
    "livdet2015_GreenBit"       : os.path.join(data_root, "livdet2015", "GreenBit"),
    "livdet2015_HiScan"         : os.path.join(data_root, "livdet2015", "HiScan")
}

for sensor_key, sensor_dir in sensors.items():
    train_paths = _filter_by_id(_collect_live_paths(sensor_dir, "Train"), rng)
    test_paths = _filter_by_id(_collect_live_paths(sensor_dir, "Test"), rng)

    id_to_paths = {}
    subject_to_paths: dict[str, list[str]] = {}
    for p in test_paths:
        subject_to_paths.setdefault(_extract_key(p, "subject"), []).append(p)
        id_to_paths.setdefault(_extract_key(p, "id"), []).append(p)

        subject_counts = {sid: len(paths) for sid, paths in subject_to_paths.items()}
        id_counts = {id_: len(paths) for id_, paths in id_to_paths.items()}

    max_subject = max(subject_counts, key=subject_counts.get)
    min_subject = min(subject_counts, key=subject_counts.get)

    max_id = max(id_counts, key=id_counts.get)
    min_id = min(id_counts, key=id_counts.get)

    max_subject_count = subject_counts[max_subject]
    min_subject_count = subject_counts[min_subject]

    max_id_count = id_counts[max_id]
    min_id_count = id_counts[min_id]

    print(sensor_key)
    print(f"Max paths: {max_subject_count} (Subject: {max_subject})")
    print(f"Min paths: {min_subject_count} (Subject: {min_subject})")
    print()
    print(f"Max paths: {max_id_count} (Id: {max_id})")
    print(f"Min paths: {min_id_count} (Id: {min_id})")
    print("-" * 20 + "\n")

livdet2011_Biometrika
Max paths: 5 (Subject: livdet2011_Biometrika_100)
Min paths: 5 (Subject: livdet2011_Biometrika_100)

Max paths: 5 (Id: livdet2011_Biometrika_100)
Min paths: 5 (Id: livdet2011_Biometrika_100)
--------------------

livdet2011_Digital
Max paths: 40 (Subject: livdet2011_Digital_5279026)
Min paths: 10 (Subject: livdet2011_Digital_0678878)

Max paths: 20 (Id: livdet2011_Digital_5279026_R1)
Min paths: 5 (Id: livdet2011_Digital_5346261_R1)
--------------------

livdet2011_Italdata
Max paths: 5 (Subject: livdet2011_Italdata_100)
Min paths: 5 (Subject: livdet2011_Italdata_100)

Max paths: 5 (Id: livdet2011_Italdata_100)
Min paths: 5 (Id: livdet2011_Italdata_100)
--------------------

livdet2011_Sagem
Max paths: 88 (Subject: livdet2011_Sagem_5297527)
Min paths: 13 (Subject: livdet2011_Sagem_4742604)

Max paths: 20 (Id: livdet2011_Sagem_2875954_R1)
Min paths: 3 (Id: livdet2011_Sagem_5539411_R5)
--------------------

livdet2013_Biometrika
Max paths: 100 (Subject: livdet2013_Bi

In [28]:
from pathlib import Path

folder_path = Path("data/SD302/")

file_count = sum(1 for f in folder_path.rglob("*") if f.is_file())

print(file_count)

25086
